# Score one node's forecasts: EMA spread

Hypothesis: a positive fast/slow EMA spread predicts positive subsequent returns. This is a workflow example, not an established edge. See `examples/EMA_RESEARCH.md`.

The node is replayed only by a backtest: `sugarloaf run --adapter quantconnect --manifest deployments/quantconnect/ema_forecasts.toml` runs the score alone as a forecast run and saves its archive to the Object Store under `sugarloaf/results/<project>/<backtest>/forecasts.json`. This notebook, run in the same project's research kernel, reads that archive, reads the prices it needs with one `qb.history` call, and scores the forecasts.

In [ ]:
import json
from datetime import UTC, datetime, timedelta
from zoneinfo import ZoneInfo

from AlgorithmImports import QuantBook, Resolution

from sugarloaf.core.primitives import InstrumentId
from sugarloaf.research import ReturnOutcomeResolver, evaluate
from sugarloaf.research.anchors import anchor_from_closes, closes_from_frame
from sugarloaf.research.serialization import decode_archive

qb = QuantBook()
# The key is printed in the backtest's SUGARLOAF_RESULT line.
KEY = "sugarloaf/results/<project>/<backtest>/forecasts.json"
archive = decode_archive(json.loads(qb.object_store.read(KEY)))
print("Forecasts:", len(archive))

Outcomes come from a separate price read. The forecasts' targets are daily price-return intervals ending at session closes, so daily raw closes stamped at the close are the anchors. An endpoint with no price (an unscheduled closure) stays unavailable; nothing is substituted.

In [ ]:
records = list(archive)
instruments = {r.view.target.subject for r in records}
first = min(r.view.target.time.start for r in records)
last = max(r.view.target.time.end for r in records)
spy = qb.add_equity("SPY", Resolution.DAILY).symbol
frame = qb.history([spy], first - timedelta(days=5), last + timedelta(days=5), Resolution.DAILY)
new_york = ZoneInfo("America/New_York")
closes = closes_from_frame(
    frame,
    instrument_of=lambda symbol: InstrumentId("USA:SPY"),
    # History stamps a daily bar at its end; make it an aware instant.
    time_of=lambda stamp: stamp.to_pydatetime().replace(tzinfo=new_york).astimezone(UTC),
)
resolver = ReturnOutcomeResolver(
    anchor_from_closes(closes), provenance={"source": "qb.history daily raw"}
)
report = evaluate(archive, resolver, as_of=datetime.now(UTC))
print("Coverage:", report.coverage)
report.summary(pool_horizons=True)

In [ ]:
[(row.forecast.view.metadata.view_id, row.status, row.metrics) for row in report.rows[:5]]

Record the source, configuration, dates, results, failed variants and limitations beside the node. The notebook ships without generated outputs.